# 🧬 HollyShit Phase 1 v2: 全掩码双头字符编码器蒸馏 (`TinyCharEncoderCUDA`)

- **双头去污染字符蒸馏架构 (Zero WeakDecoder CE)**:
  - **冻结五师神之流形教师 (`SensoryFuser + GodEncoder`)**: 联合监督子句语义规划头 `z_god`（含五师中心化余弦关系矩阵蒸馏 + 词序打乱硬负例 `OrderDirectionHead` + 非对称顺向语篇推进 `AsymmetricNextClauseHead`）；
  - **原生正交 32-D 字符锚点子空间头 (`char_subspace_proj`)**: 全链路强制 `<PAD>` 注意力掩码与隐状态清零，配合有效字符级 VICReg 各向同性均衡 + Stiefel 正交约束 + 字符/二元转移探针，彻底根除 `<PAD>` 污染与 `93.5%` PC1 方差垄断。
- **统一多数据源管道 (`gdrive` 默认 / `ms` 魔搭直连 / `hf` CDN / `local` 本地目录 + 残损分块在线自愈)**:
  - **`gdrive` (默认云盘持久源)**: 自动将 `.npz` 分块按需暂存至本地高速 NVMe `/content/embs_cache`，配合 `--no-prefetch` 单块内存驻留（峰值内存约 `13.5~15.0 GB`，含 Parquet 文本池 + 单块特征），消费完立即清除本地暂存，云盘健康原始文件 100% 零修改、零误删；若云盘尚无分块则自动无感回退 `ms` 流式源，若遇残损/缺失单块则自动远端拉取并原子修复回写云盘；
  - **`ms` / `hf` / `local`**: 支持 ModelScope (`catclever/emb_npy`)、Hugging Face (`drunkencat/emb_npy`) 按需流式拉取或本地目录直读。
- **双重防丢权硬门禁 (`Mount Guard + Crash-Guard + HF Off-Drive Mirror`)**:
  - 每个关键单元格与训练主程序第 0 秒强制校验 `os.path.ismount('/content/drive')` 及真实挂载表，每 `--save_steps 2000` 高频存档（`--keep_last_ckpts 3` 滚动保留最近 3 份）并支持 `--auto_resume` 断点续训与异常退出 `latest_emergency` 抢救；
  - 收尾自动导出 `charencoder_pt_v2_best/phase1_charencoder.safetensors`（供下游 Phase 3 `Translator` 与 Phase 4 左脑直接绑定），并可选镜像备份至 Hugging Face 私有模型仓。
- **推荐 Colab 运行时规格**: **A100 (`Batch=1024`) / L4 (`Batch=512`) + 高内存 (`High-RAM >= 25GB`)**。

In [ ]:
# 0. [可选] 清理旧版 Phase 1 v2 Checkpoint (仅当确认 Google Drive 已挂载且需从零重训时取消注释；默认保留以支持 --auto_resume 断点续训)
# import os; assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！'
# !rm -rf /content/drive/MyDrive/hollyshit/checkpoints/p1_v2/charencoder_pt_v2_*


In [ ]:
import os
import shutil
from google.colab import drive

# 1. 挂载 Google Drive 并立即执行硬门禁校验 (防止切换运行时后未挂载写进临时盘!)
drive.mount('/content/drive', force_remount=False)
assert os.path.ismount('/content/drive'), '❌ Google Drive 未挂载成功！请授权后再运行后续单元格！'

# 2. 安装 Hugging Face 高速流式下载 (hf_transfer)、魔搭源、权重序列化与 Parquet 解析依赖
!pip install -q "huggingface_hub[hf_transfer]" modelscope safetensors pyarrow pandas || pip install -q huggingface_hub modelscope safetensors pyarrow pandas

try:
    import hf_transfer  # noqa: F401
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    print('⚡ 已启用 hf_transfer Rust 高速 CDN 多连接加速 (HF_HUB_ENABLE_HF_TRANSFER=1)')
except ImportError:
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
    print('ℹ️ 未检测到 hf_transfer 轮子，已安全使用标准 huggingface_hub 下载器')

# 3. 读取 Colab Secrets 中的 HF_TOKEN (及可选 MODELSCOPE_API_TOKEN)，自动激活数据集与私有权重双活备份仓
os.environ.setdefault('HF_CKPT_REPO', '')
try:
    from google.colab import userdata
    hf_tok = userdata.get('HF_TOKEN')
    if hf_tok and hf_tok.strip():
        hf_tok = hf_tok.strip()
        os.environ['HF_TOKEN'] = hf_tok
        from huggingface_hub import HfApi
        hf_user = HfApi(token=hf_tok).whoami()['name']
        hf_ds_repo = f'{hf_user}/emb_npy'
        hf_ckpt_repo = f'{hf_user}/hollyshit-checkpoints'
        os.environ['HF_DATASET_REPO'] = hf_ds_repo
        os.environ['HF_CKPT_REPO'] = hf_ckpt_repo
        print(f'🔑 已加载 HF_TOKEN (用户: {hf_user})')
        print(f'   • 优先 HF 数据集源 : {hf_ds_repo}')
        print(f'   • 异地私有权重备份 : {hf_ckpt_repo}')
except Exception as _hf_err:
    print(f'ℹ️ 未配置或无法解析 HF_TOKEN (将探测公开 HF 源或魔搭源，跳过 HF 私有权重镜像): {_hf_err}')

try:
    from google.colab import userdata
    ms_tok = userdata.get('MODELSCOPE_API_TOKEN')
    if ms_tok and ms_tok.strip():
        os.environ['MODELSCOPE_API_TOKEN'] = ms_tok.strip()
        print('🔑 已从 Colab Secrets 加载 MODELSCOPE_API_TOKEN')
except Exception:
    pass

# 4. 硬件规格与高内存 (High-RAM) 安全自检
import torch

def _get_system_ram_gb() -> float:
    if os.path.exists('/proc/meminfo'):
        with open('/proc/meminfo', 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('MemTotal:'):
                    kb = int(line.split()[1])
                    return kb / (1024.0 ** 2)
    return 0.0

ram_gb = _get_system_ram_gb()
disk_free_gb = shutil.disk_usage('/content').free / (1024.0 ** 3)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (未检测到 GPU!)'
vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3)) if torch.cuda.is_available() else 0.0
drive_mounted = os.path.ismount('/content/drive')

print('\n' + '=' * 62)
print(f'🔥 GPU 设备   : {gpu_name} ({vram_gb:.1f} GB VRAM)')
print(f'🧠 系统内存   : {ram_gb:.1f} GB RAM')
print(f'💾 本地 SSD   : /content 可用空间 {disk_free_gb:.1f} GB')
print(f'📂 云盘挂载   : /content/drive ismount={drive_mounted}')
print('=' * 62)
if ram_gb > 0 and ram_gb < 20.0:
    print('⚠️ [重要警告] 当前系统内存 < 20 GB！Phase 1 v2 需同时驻留 Parquet 文本池 + 单块 50w 五师特征 (峰值约 13.5~15 GB RAM)。')
    print('   请在菜单栏「代码执行程序 -> 更改运行时类型」中开启【高内存 (High-RAM >= 25GB)】后再启动训练！')


In [ ]:
import os
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！严禁在本地容器盘创建伪目录！'

# 从 GitHub 拉取纯代码并初始化 training/core 子模块 (如已存在则 git pull)
if not os.path.exists('/content/HollyShit'):
    !git clone https://github.com/catclever/HollyShit.git /content/HollyShit
    %cd /content/HollyShit
    !git submodule update --init training/core
else:
    %cd /content/HollyShit
    !git pull
    !git submodule update --init training/core

# 再次硬校验挂载点后，创建真实 Google Drive Checkpoint、数据与向量存储目录并建立软链接
assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！严禁在本地容器盘创建伪目录！'
os.makedirs('/content/drive/MyDrive/hollyshit/checkpoints/p1_v2', exist_ok=True)
os.makedirs('/content/drive/MyDrive/hollyshit/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/hollyshit/embs', exist_ok=True)
!ln -sfn /content/drive/MyDrive/hollyshit/checkpoints checkpoints
!ln -sfn /content/drive/MyDrive/hollyshit/data data

# 数据源与上游教师配置：默认优先复用云盘已落盘的 Phase 0 v3 权重与 gdrive .npz 分块 (若云盘无分块则在下一格自动回退 ms)
DATA_SOURCE = "gdrive"
GDRIVE_DIR = "/content/drive/MyDrive/hollyshit/embs"
P0_CKPT_DIR = "/content/drive/MyDrive/hollyshit/checkpoints/p0_v3"
os.environ['DATA_SOURCE'] = DATA_SOURCE
os.environ['GDRIVE_DIR'] = GDRIVE_DIR
os.environ['P0_CKPT_DIR'] = P0_CKPT_DIR

print('✅ 仓库与真实 Google Drive 云盘软链接就绪:', os.getcwd())
print(f'⚙️ 当前配置: DATA_SOURCE={DATA_SOURCE!r}, GDRIVE_DIR={GDRIVE_DIR!r}, P0_CKPT_DIR={P0_CKPT_DIR!r}')


In [ ]:
import argparse
import glob
import json
import os
import re
import sys
import torch
from huggingface_hub import HfApi

# 1. 确保工作目录在 /content/HollyShit，并创建本地高速 NVMe SSD 分块缓存目录 (严禁放在 Google Drive!)
assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！请先挂载 Google Drive！'
if os.path.isdir('/content/HollyShit'):
    os.chdir('/content/HollyShit')
if '/content/HollyShit' not in sys.path:
    sys.path.insert(0, '/content/HollyShit')
local_embs_cache = '/content/embs_cache'
os.makedirs(local_embs_cache, exist_ok=True)

from training.data.alignment_dataloader import (
    is_valid_npz_archive,
    repair_gdrive_chunk_from_remote,
)
from training.train_phase1_charencoder_cuda import (
    infer_teacher_arch_from_ckpt,
    resolve_or_fetch_parquet_path,
    resolve_p0_teacher_ckpt_dir,
)

# 加载或自愈生成 dataset_config.json
if not os.path.exists('dataset_config.json'):
    default_ds_cfg = {
        'default_source': 'ms',
        'modelscope_repo_id': 'catclever/emb_npy',
        'huggingface_repo_id': 'drunkencat/emb_npy',
        'gdrive_dir': '/content/drive/MyDrive/hollyshit/embs',
        'base_models': ['bge', 'qwen', 'xiaobu', 'youtu', 'conan_v1'],
        'chunk_name_patterns': {
            'bge': 'bge_m3/bge_embeddings_chunk_{start:07d}_{end:07d}.npz',
            'qwen': 'gte_qwen/gte_qwen2_embeddings_chunk_{start:07d}_{end:07d}.npz',
            'xiaobu': 'xiaobu/xiaobu_embeddings_chunk_{start:07d}_{end:07d}.npz',
            'youtu': 'youtu/youtu_embeddings_chunk_{start:07d}_{end:07d}.npz',
            'conan_v1': 'conan/conan_v1_embeddings_chunk_{start:07d}_{end:07d}.npz',
        },
        'emb_dims': [1024, 3584, 1792, 2048, 1792],
        'parquet_path': 'data/Basic_ZH/chunked_mixed_omni.parquet',
    }
    with open('dataset_config.json', 'w', encoding='utf-8') as _wf:
        json.dump(default_ds_cfg, _wf, indent=4, ensure_ascii=False)
    print('🛠️ 已自动生成默认 dataset_config.json')

with open('dataset_config.json', 'r', encoding='utf-8') as f:
    ds_cfg = json.load(f)

models = ds_cfg['base_models']
patterns = ds_cfg['chunk_name_patterns']
cand_hf_repo = os.environ.get('HF_DATASET_REPO') or ds_cfg.get('huggingface_repo_id', 'drunkencat/emb_npy')
ms_repo_id = ds_cfg.get('modelscope_repo_id', 'catclever/emb_npy')
active_data_source = os.environ.get('DATA_SOURCE', ds_cfg.get('default_source', 'gdrive'))
active_gdrive_dir = os.environ.get('GDRIVE_DIR', ds_cfg.get('gdrive_dir', '/content/drive/MyDrive/hollyshit/embs'))
p0_ckpt_input = os.environ.get('P0_CKPT_DIR', '/content/drive/MyDrive/hollyshit/checkpoints/p0_v3')

# 2. 自动解析并严格校验冻结的 Phase 0 v3 教师检查点 (支持 best 目录或最新 step_* 目录自动发现)
resolved_p0 = resolve_p0_teacher_ckpt_dir(p0_ckpt_input)
_fuser_p, _god_p, _t_dims, _t_dmodel, _t_zdim = infer_teacher_arch_from_ckpt(resolved_p0)
assert os.path.isdir(resolved_p0), f'❌ 未找到有效的 Phase 0 v3 教师检查点目录: {resolved_p0}'
os.environ['ACTIVE_P0_CKPT'] = resolved_p0
print(f'🏛️ 已锁定冻结 Phase 0 v3 教师检查点: {resolved_p0}')
print(f'   • 教师路由结构 : {len(_t_dims)} 路 {_t_dims} -> d_model={_t_dmodel} -> z_dim={_t_zdim}')
_p0_ms_path = os.path.join(resolved_p0, 'manifold_stats.pt')
if os.path.isfile(_p0_ms_path) and os.path.getsize(_p0_ms_path) > 0:
    _p0_ms = torch.load(_p0_ms_path, map_location='cpu', weights_only=False)
    _p0_hm = _p0_ms.get('health_metrics', {})
    _p0_step = _p0_ms.get('global_step')
    _p0_sphere = _p0_ms.get('sphere_scale')
    print(f'   • 教师训练步数 : global_step={_p0_step} (sphere_scale={_p0_sphere})')
    if _p0_hm:
        _pr = _p0_hm.get('participation_ratio', 0.0)
        _rcos = _p0_hm.get('mean_teacher_recon_cos', 0.0)
        _rspear = _p0_hm.get('relational_spearman', 0.0)
        print(f'   • 教师流形体检 : PR={_pr:.1f} | ReconCos={_rcos:.4f} | RelSpearman={_rspear:.4f}')

# 3. 校验或自动拉取基准子句文本语料 chunked_mixed_omni.parquet
_target_parquet = '/content/drive/MyDrive/hollyshit/data/Basic_ZH/chunked_mixed_omni.parquet'
_probe_args = argparse.Namespace(
    gdrive_dir=active_gdrive_dir,
    data_dir=local_embs_cache,
    download_cache_dir=local_embs_cache,
    ms_repo_id=ms_repo_id,
    hf_repo_id=cand_hf_repo,
)
resolved_parquet = resolve_or_fetch_parquet_path(_target_parquet, _probe_args, ds_cfg)
if not (os.path.isfile(resolved_parquet) and os.path.getsize(resolved_parquet) > 0):
    raise FileNotFoundError(
        '❌ 未找到 Phase 1 v2 必需的子句文本语料文件 chunked_mixed_omni.parquet！\n'
        '👉 请将该文件上传至 Google Drive 路径: /content/drive/MyDrive/hollyshit/data/Basic_ZH/chunked_mixed_omni.parquet 后重试！'
    )
_pq_mb = os.path.getsize(resolved_parquet) / (1024.0 ** 2)
print(f'📖 已就绪对齐文本语料 Parquet: {resolved_parquet} ({_pq_mb:.1f} MB)')

# 4. 校验选定数据源的 .npz 分块完整性 (gdrive 为空时自动无感切换 ms；发现残损/缺失单块时自动在线自愈)
def _count_aligned_chunks(file_list):
    per_m = {}
    for m_name in models:
        prefix = patterns[m_name].split('/')[0]
        bounds = {}
        for rf in file_list:
            rf_norm = rf.replace('\\', '/')
            if rf_norm.endswith('.npz') and (rf_norm.startswith(prefix + '/') or f'/{prefix}/' in rf_norm):
                mm = re.search(r'chunk_(\d+)_(\d+)\.npz', os.path.basename(rf_norm))
                if mm:
                    bounds[int(mm.group(1))] = int(mm.group(2))
        per_m[m_name] = bounds
    common = sorted(set.intersection(*[set(b.keys()) for b in per_m.values()])) if per_m else []
    clauses = min(per_m[m][common[-1]] for m in models) if common else 0
    return per_m, common, clauses

print(f'📡 当前选择的数据源: {active_data_source.upper()}')
if active_data_source == 'gdrive':
    assert os.path.ismount('/content/drive'), '❌ 使用 gdrive 源必须首先挂载 /content/drive！'

    def _scan_non_temp_gdrive_npz(gd_root):
        found = []
        for _p in glob.glob(os.path.join(gd_root, '**', '*.npz'), recursive=True):
            _p_norm = _p.replace('\\', '/')
            _p_base = os.path.basename(_p_norm)
            if (
                '/._____temp/' in _p_norm
                or _p_norm.startswith('._____temp/')
                or '/.cache/' in _p_norm
                or _p_norm.startswith('.cache/')
                or '/_npz_repair_staging/' in _p_norm
                or _p_norm.startswith('_npz_repair_staging/')
                or _p_base.endswith(('.incomplete', '.tmp', '.partial'))
                or '.tmp.' in _p_base
                or '.repair_' in _p_base
                or not _p_base.endswith('.npz')
            ):
                continue
            if os.path.isfile(_p):
                found.append(_p)
        return found

    def _infer_cand_name_from_path(_p):
        _p_norm = _p.replace('\\', '/')
        _mm = re.search(r'chunk_(\d+)_(\d+)\.npz', os.path.basename(_p_norm))
        if not _mm:
            return None
        _s_idx, _e_idx = int(_mm.group(1)), int(_mm.group(2))
        for _m_name in models:
            _pref = patterns[_m_name].split('/')[0]
            if _p_norm.startswith(_pref + '/') or f'/{_pref}/' in _p_norm:
                return patterns[_m_name].format(start=_s_idx, end=_e_idx)
        return None

    _all_gd_npz = _scan_non_temp_gdrive_npz(active_gdrive_dir)
    if len(_all_gd_npz) == 0:
        print(f'ℹ️ 云盘目录 {active_gdrive_dir} 中尚未存放离线 .npz 分块，自动切换至 ModelScope (ms) 按需流式拉取源！')
        active_data_source = 'ms'
        DATA_SOURCE = 'ms'
        os.environ['DATA_SOURCE'] = 'ms'
        os.environ['ACTIVE_HF_REPO'] = ''
    else:
        gdrive_files = []
        _repair_plan = []
        _queued_cands = set()
        for _gf in _all_gd_npz:
            if is_valid_npz_archive(_gf):
                gdrive_files.append(_gf)
            else:
                _inferred_cand = _infer_cand_name_from_path(_gf)
                if _inferred_cand and _inferred_cand not in _queued_cands:
                    _repair_plan.append((_inferred_cand, _gf))
                    _queued_cands.add(_inferred_cand)

        gd_per_model, gd_common, gd_clauses = _count_aligned_chunks(gdrive_files)
        _union_starts = sorted(set.union(*[set(b.keys()) for b in gd_per_model.values()])) if gd_per_model else []
        for _s in _union_starts:
            _end_val = min(gd_per_model[_m][_s] for _m in models if _s in gd_per_model[_m])
            for _m in models:
                if _s not in gd_per_model[_m]:
                    _miss_cand = patterns[_m].format(start=_s, end=_end_val)
                    if _miss_cand not in _queued_cands:
                        _repair_plan.append((_miss_cand, None))
                        _queued_cands.add(_miss_cand)

        if _repair_plan:
            print(f'⚠️ [GDrive 预检发现异常] 检测到 {len(_repair_plan)} 个残损或缺失分块，正在自动拉取修复:')
            for _r_idx, (_cand_fn, _broken_fp) in enumerate(_repair_plan, 1):
                _reason = f'损坏文件 ({_broken_fp})' if _broken_fp else '缺失教师分块'
                print(f'   [{_r_idx}/{len(_repair_plan)}] {_cand_fn} -> {_reason}')
                _staged_out = repair_gdrive_chunk_from_remote(
                    active_gdrive_dir,
                    _cand_fn,
                    cache_dir=local_embs_cache,
                    ms_repo_id=ms_repo_id,
                    hf_repo_id=cand_hf_repo,
                    broken_gdrive_path=_broken_fp,
                )
                if os.path.isfile(_staged_out) and os.path.abspath(_staged_out).startswith(
                    os.path.abspath(local_embs_cache) + os.sep
                ):
                    try:
                        os.remove(_staged_out)
                    except OSError:
                        pass
            gdrive_files = [_p for _p in _scan_non_temp_gdrive_npz(active_gdrive_dir) if is_valid_npz_archive(_p)]
            gd_per_model, gd_common, gd_clauses = _count_aligned_chunks(gdrive_files)
            print(f'✅ [GDrive 自愈完成] 全部 {len(gdrive_files)} 个 .npz 分块 ZIP 尾部索引与矩阵头 100% 完整无损！')
        else:
            print(f'✅ [GDrive 体检通过] 全部 {len(gdrive_files)} 个 .npz 分块 ZIP 尾部索引与矩阵头 100% 完整无损！')

        print(f'📂 [GDrive] 目录: {active_gdrive_dir}，探测到 {len(gd_common)} 个对齐宏分块 ({gd_clauses:,} 条子句)')
        os.environ['ACTIVE_HF_REPO'] = ''
elif active_data_source == 'hf':
    try:
        hf_files = list(HfApi(token=os.environ.get('HF_TOKEN') or None).list_repo_files(repo_id=cand_hf_repo, repo_type='dataset'))
        hf_per_model, hf_common, hf_clauses = _count_aligned_chunks(hf_files)
        os.environ['ACTIVE_HF_REPO'] = cand_hf_repo
        print(f'✅ Hugging Face 源 [{cand_hf_repo}]: 共 {len(hf_common)} 个对齐宏分块，总计 {hf_clauses:,} 条子句。')
    except Exception as _e:
        print(f'⚠️ Hugging Face 源不可用: {_e}，回退到 ModelScope')
        active_data_source = 'ms'
        DATA_SOURCE = 'ms'
        os.environ['DATA_SOURCE'] = 'ms'
        os.environ['ACTIVE_HF_REPO'] = ''

if active_data_source == 'ms':
    from modelscope.hub.api import HubApi
    ms_api = HubApi()
    if os.environ.get('MODELSCOPE_API_TOKEN'):
        try:
            ms_api.login(os.environ['MODELSCOPE_API_TOKEN'])
        except Exception as e:
            print(f'⚠️ ModelScope Token 鉴权跳过: {e}')
    ms_files = [item.get('Path', item.get('Name', '')) for item in ms_api.get_dataset_files(ms_repo_id, recursive=True)]
    ms_per_model, ms_common, ms_clauses = _count_aligned_chunks(ms_files)
    assert len(ms_common) > 0, '未在魔搭探测到分块，请检查网络！'
    os.environ['ACTIVE_HF_REPO'] = ''
    print(f'✅ 已启用 ModelScope 流式源 [{ms_repo_id}]！共 {len(ms_common)} 个对齐宏分块，总计 {ms_clauses:,} 条子句。')

# 5. 按 GPU 档位 (A100 vs L4 vs T4) 自适应设定 6 层 Transformer + 双路词序硬负例最优超参
vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3)) if torch.cuda.is_available() else 0.0
ram_gb = 0.0
if os.path.exists('/proc/meminfo'):
    with open('/proc/meminfo', 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith('MemTotal:'):
                ram_gb = int(line.split()[1]) / (1024.0 ** 2)
                break

if vram_gb >= 38.0 and ram_gb >= 24.0:
    p1_cfg = {
        'P1_BATCH_SIZE': '1024',
        'P1_LR': '2e-4',
        'P1_WARMUP': '1500',
    }
    tier_label = 'A100 旗舰满血档 (Batch=1024, LR=2e-4)'
elif vram_gb >= 22.0 and ram_gb >= 24.0:
    p1_cfg = {
        'P1_BATCH_SIZE': '512',
        'P1_LR': '1.5e-4',
        'P1_WARMUP': '2500',
    }
    tier_label = 'L4 高吞吐主力档 (Batch=512, LR=1.5e-4)'
else:
    p1_cfg = {
        'P1_BATCH_SIZE': '256',
        'P1_LR': '1e-4',
        'P1_WARMUP': '4000',
    }
    tier_label = 'T4 / 标准稳健档 (Batch=256, LR=1e-4)'

os.environ.update(p1_cfg)
os.environ.setdefault('HF_CKPT_REPO', '')
_active_ds_env = os.environ.get('DATA_SOURCE')
_active_hf_env = os.environ.get('ACTIVE_HF_REPO')
print(f'⚙️ 已激活 Phase 1 v2 训练配置 [{tier_label}]: {p1_cfg} | DATA_SOURCE={_active_ds_env!r} | ACTIVE_HF_REPO={_active_hf_env!r}')


In [ ]:
# ==============================================================================
# （可选）直接下载分块至 Google Drive（存好后将 DATA_SOURCE 设为 'gdrive' 即可完全离线/零外部网络依赖极速训练）
# - 使用 ModelScope dataset_file_download 直连极速下载到 Google Drive 持久目录
# - 自动执行 is_valid_npz_archive 完整性校验，跳过真正无损的分块并自动重下被截断的残损分块
# ==============================================================================
DOWNLOAD_TO_GDRIVE = False  # 若需要将全部或部分分块预先保存在 Google Drive，请改为 True

if DOWNLOAD_TO_GDRIVE:
    assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！'
    import glob
    import json
    import os
    import sys
    from modelscope.hub.file_download import dataset_file_download
    from modelscope.hub.api import HubApi
    if '/content/HollyShit' not in sys.path:
        sys.path.insert(0, '/content/HollyShit')
    from training.data.alignment_dataloader import (
        is_valid_npz_archive,
        validate_npz_archive,
    )

    os.makedirs(GDRIVE_DIR, exist_ok=True)
    with open('dataset_config.json', 'r', encoding='utf-8') as f:
        _cfg = json.load(f)

    _ms_repo = _cfg.get('modelscope_repo_id', 'catclever/emb_npy')

    print(f'📥 开始同步 ModelScope ({_ms_repo}) 分块至 Google Drive: {GDRIVE_DIR} ...')
    _api = HubApi()
    if os.environ.get('MODELSCOPE_API_TOKEN'):
        try:
            _api.login(os.environ['MODELSCOPE_API_TOKEN'])
        except Exception:
            pass
    _files = [item.get('Path', item.get('Name', '')) for item in _api.get_dataset_files(_ms_repo, recursive=True)]
    _npz_files = [f for f in _files if f.endswith('.npz')]
    print(f'📦 远端共检测到 {len(_npz_files)} 个 .npz 分块文件。')

    _existing_keys = set()
    _corrupt_gd_map = {}
    _abs_gd = os.path.abspath(GDRIVE_DIR)
    for _ep in glob.glob(os.path.join(GDRIVE_DIR, '**', '*.npz'), recursive=True):
        _ep_norm = _ep.replace('\\', '/')
        _ep_base = os.path.basename(_ep_norm)
        if (
            '/._____temp/' in _ep_norm
            or _ep_norm.startswith('._____temp/')
            or '/.cache/' in _ep_norm
            or _ep_norm.startswith('.cache/')
            or '/_npz_repair_staging/' in _ep_norm
            or _ep_norm.startswith('_npz_repair_staging/')
            or _ep_base.endswith(('.incomplete', '.tmp', '.partial'))
            or '.tmp.' in _ep_base
            or '.repair_' in _ep_base
            or not _ep_base.endswith('.npz')
        ):
            continue
        if not os.path.isfile(_ep):
            continue
        _rel = os.path.relpath(os.path.abspath(_ep), _abs_gd).replace('\\', '/')
        _parts = _rel.split('/')
        if is_valid_npz_archive(_ep):
            _existing_keys.add(_rel)
            if len(_parts) >= 2:
                _tail_key = f'{_parts[-2]}/{_parts[-1]}'
                _existing_keys.add(_tail_key)
            _existing_keys.add(_parts[-1])
        else:
            _corrupt_gd_map[_rel] = os.path.abspath(_ep)
            if len(_parts) >= 2:
                _tail_key = f'{_parts[-2]}/{_parts[-1]}'
                _corrupt_gd_map.setdefault(_tail_key, os.path.abspath(_ep))
            _corrupt_gd_map.setdefault(_parts[-1], os.path.abspath(_ep))

    for idx, remote_fn in enumerate(_npz_files, 1):
        _r_norm = remote_fn.replace('\\', '/').lstrip('/')
        _r_parts = _r_norm.split('/')
        _r_tail2 = f'{_r_parts[-2]}/{_r_parts[-1]}' if len(_r_parts) >= 2 else _r_norm
        _r_base = _r_parts[-1]
        target_dest = os.path.join(GDRIVE_DIR, remote_fn)
        if (
            (os.path.exists(target_dest) and is_valid_npz_archive(target_dest))
            or _r_norm in _existing_keys
            or _r_tail2 in _existing_keys
            or _r_base in _existing_keys
        ):
            continue
        if os.path.isfile(target_dest) and not is_valid_npz_archive(target_dest):
            print(f'  🧹 清理残损云盘文件后重下: {target_dest}')
            os.remove(target_dest)
        for _ck in (_r_norm, _r_tail2, _r_base):
            _bad_p = _corrupt_gd_map.pop(_ck, None)
            if _bad_p and os.path.isfile(_bad_p) and not is_valid_npz_archive(_bad_p):
                print(f'  🧹 清理残损嵌套云盘文件后重下: {_bad_p}')
                os.remove(_bad_p)
        print(f'  [{idx}/{len(_npz_files)}] 正在下载: {remote_fn} -> {GDRIVE_DIR} ...')
        try:
            downloaded = dataset_file_download(_ms_repo, remote_fn, local_dir=GDRIVE_DIR, cache_dir=GDRIVE_DIR)
        except TypeError:
            downloaded = dataset_file_download(_ms_repo, remote_fn, cache_dir=GDRIVE_DIR)
        validate_npz_archive(downloaded)
        _existing_keys.update([_r_norm, _r_tail2, _r_base])
        print(f'  ✅ 已保存并通过完整性校验: {downloaded}')
    print('🎉 Google Drive 离线分块同步完毕！请将 DATA_SOURCE 设为 "gdrive" 开启完全离线训练！')


In [ ]:
# ==============================================================================
# 🚀 启动 Phase 1 v2: 全掩码双头字符编码器蒸馏 (TinyCharEncoderCUDA)
# - 启动前双重硬门禁校验 os.path.ismount('/content/drive')，杜绝未挂载静默写临时盘
# - --p0_ckpt "$ACTIVE_P0_CKPT": 冻结 Phase 0 v3 神之流形教师 (Zero WeakDecoder CE)
# - --data_source $DATA_SOURCE & --gdrive_dir "$GDRIVE_DIR" & --no-prefetch: 单块 NVMe 控流
# - --save_steps 2000 & --keep_last_ckpts 3 & --auto_resume: 高频滚动存档 + 断点自动续训
# ==============================================================================
import os
assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！已紧急拦截训练启动，请先挂载 Google Drive！'

%cd /content/HollyShit
!python -u training/train_phase1_charencoder_cuda.py \
  --p0_ckpt "$ACTIVE_P0_CKPT" \
  --config_file dataset_config.json \
  --data_source $DATA_SOURCE \
  --gdrive_dir "$GDRIVE_DIR" \
  --hf_repo_id "$ACTIVE_HF_REPO" \
  --hf_ckpt_repo "$HF_CKPT_REPO" \
  --data_dir /content/embs_cache \
  --download_cache_dir /content/embs_cache \
  --no-prefetch \
  --batch_size $P1_BATCH_SIZE \
  --epochs 3 \
  --lr $P1_LR \
  --warmup_steps $P1_WARMUP \
  --out_dir /content/drive/MyDrive/hollyshit/checkpoints/p1_v2 \
  --ckpt_prefix charencoder_pt_v2 \
  --eval_steps 200 \
  --save_steps 2000 \
  --keep_last_ckpts 3 \
  --auto_resume


In [ ]:
import os
import shutil
import sys
import time
import torch
from google.colab import drive

# 1. 收尾前首先硬校验 /content/drive 确实处于挂载状态
assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！无法校验或安全刷写云盘检查点！'

if '/content/HollyShit' not in sys.path:
    sys.path.insert(0, '/content/HollyShit')
from training.train_phase1_charencoder_cuda import (
    mirror_checkpoint_to_hf_hub,
    resolve_latest_phase1_v2_checkpoint,
)

drive_ckpt_root = '/content/drive/MyDrive/hollyshit/checkpoints/p1_v2'
ckpt_prefix = 'charencoder_pt_v2'
latest_ckpt = resolve_latest_phase1_v2_checkpoint(drive_ckpt_root, ckpt_prefix)
assert latest_ckpt is not None, f'❌ 未在云盘目录 {drive_ckpt_root} 中检测到任何有效检查点！'

print('=' * 68)
print(f'🏆 最新有效 Phase 1 v2 检查点: {latest_ckpt}')
stats_path = os.path.join(latest_ckpt, 'char_subspace_stats.pt')
if os.path.isfile(stats_path):
    st = torch.load(stats_path, map_location='cpu', weights_only=False)
    hm = st.get('health_metrics', {})
    st_step = st.get('global_step')
    st_dmodel = st.get('d_model')
    st_dsub = st.get('d_sub')
    print(f'   • global_step : {st_step}')
    print(f'   • arch        : d_model={st_dmodel} | d_sub={st_dsub}')
    if hm:
        distill_cos = hm.get('mean_distill_cos', hm.get('z_god_cos', 0.0))
        spearman_v = hm.get('relational_spearman', 0.0)
        order_acc = hm.get('order_pair_acc', hm.get('order_discrim_acc', 0.0))
        iso_dev = hm.get('char_subspace_ortho_dev', 0.0)
        anchor_top1 = hm.get('char_anchor_top1_acc', hm.get('subspace_char_top1_acc', 0.0))
        pc1_ratio = hm.get('subspace_pc1_var_ratio', 0.0)
        sub_pr = hm.get('subspace_participation_ratio', 0.0)
        print(
            f'   • health      : DistillCos={distill_cos:.4f} | '
            f'Spearman={spearman_v:.4f} | '
            f'OrderAcc={order_acc:.4f} | '
            f'CharIsoDev={iso_dev:.4f} | '
            f'AnchorTop1={anchor_top1:.4f} | '
            f'PC1={pc1_ratio:.4f} | PR={sub_pr:.1f}'
        )

# 2. 完整同步全部训练与下游绑定产物至 charencoder_pt_v2_best
all_export_artifacts = [
    'phase1_charencoder.safetensors',
    'student.safetensors',
    'student.pt',
    'char_subspace_stats.pt',
    'order_head.pt',
    'next_clause_head.pt',
    'char_anchor_head.pt',
    'training_args.json',
    'dataloader.json',
]
best_alias_dir = os.path.join(drive_ckpt_root, f'{ckpt_prefix}_best')
os.makedirs(best_alias_dir, exist_ok=True)
for fn in all_export_artifacts:
    src_file = os.path.join(latest_ckpt, fn)
    if os.path.isfile(src_file):
        shutil.copy2(src_file, os.path.join(best_alias_dir, fn))

# 硬校验下游 Translator / Phase 4 左脑必需的关键落盘文件存在且字节数 > 0
for must_fn in ('phase1_charencoder.safetensors', 'student.pt', 'char_subspace_stats.pt'):
    must_path = os.path.join(best_alias_dir, must_fn)
    assert os.path.isfile(must_path) and os.path.getsize(must_path) > 0, (
        f'❌ 关键权重文件缺失或为空: {must_path}'
    )
print(f'🔗 已完整同步 charencoder_pt_v2_best (共 {len(os.listdir(best_alias_dir))} 个文件): {best_alias_dir}')

# 若配置了 HF_TOKEN / HF_CKPT_REPO，同步将 charencoder_pt_v2_best 异地备份至 Hugging Face 私有模型库
hf_ckpt_repo = os.environ.get('HF_CKPT_REPO', '').strip()
if hf_ckpt_repo:
    mirror_checkpoint_to_hf_hub(best_alias_dir, hf_ckpt_repo)
print('=' * 68)

# 3. 切回 /content 释放软链接目录句柄后，强制刷写并卸载 Google Drive FUSE 缓存 (严禁吞掉异常!)
os.chdir('/content')
print('🔄 已切回 /content，正在执行 drive.flush_and_unmount() 强制刷盘...')
flush_ok = False
drive.flush_and_unmount()
flush_ok = not os.path.ismount('/content/drive')
assert flush_ok, '❌ drive.flush_and_unmount() 后 /content/drive 仍处于挂载态，请勿销毁实例！'
print('✅ Google Drive FUSE 写入缓存已 100% 安全落盘并完成卸载！')

# ⚠️ 安全铁律：默认禁用 runtime.unassign()！请先在 Google Drive 网页端确认 charencoder_pt_v2_best 权重已存在。
# 如需开启无人值守自动释放实例，必须保留 `if flush_ok:` 硬门禁（严禁在 flush 失败时销毁虚拟机）：
# if flush_ok:
#     from google.colab import runtime
#     time.sleep(5)
#     runtime.unassign()
